# board

> Class for Sudoku board

In [1]:
#| default_exp board

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export

import numpy as np
import time

from sudoku.cell import House, Cell

In [4]:
#| export

class Board:
    def __init__(self, initial_cells):
        self.log = []
        self.load_initial_cells(initial_cells)
    def __getitem__(self, key):
        return self.cells[key]
    def show(self, with_candidates=True):
        res = ""
        for r in range(-1, 10):
            for c in range(9):
                if r == -1 or r == 9:
                    res += f"{' ' if c == 0 else ''}|{c}--{'---------' if with_candidates else ''}"
                else:
                    res += f"{r if c == 0 else ''}|{self.cells[r,c].show(with_candidates)}"
            res += '|\n'
        return res
    def show_log(self):
        for i, msg in enumerate(self.log): print(i, msg)
    def clone(self):
        import copy
        return copy.deepcopy(self)
    def export(self):
        tmp1 = [x.get_value() for x in self.cells.flatten()]
        tmp2 = [str(x) if x != None else ' ' for x in tmp1]
        return ''.join(tmp2)

    def load_initial_cells(self, initial_cells):
        self.cells = np.empty((9, 9), dtype=object)
        if not isinstance(initial_cells, list):
            raise TypeError("initial_cells must be a Python list")        
        if len(initial_cells) != 9 :
            raise TypeError("initial_cells must contain 9 rows")  
        for r in range(9):
            if len(initial_cells[r]) != 9 :
                raise TypeError(f"row {r} in initial_cells must contain 9 characters") 
        for r in range(9):
            row = initial_cells[r]
            for c in range(9):
                value = None if row[c] == '-' else int(row[c])
                self.cells[r,c] = Cell(r, c, value)
        self.remove_obvious_candidates()        

    def _cells_in_row(self, row): return self.cells[row, :]
    def _cells_in_col(self, col): return self.cells[:, col]
    def _cells_in_block(self, block):
        horiz = block // 3
        ver = block % 3
        return self.cells[horiz*3:horiz*3+3, ver*3:ver*3+3].flatten()
        
    def remove_obvious_candidates(self):
        for v in range(1, 10):
            self._remove_obvious_candidates_for_value(v)
        self.check_is_ok()
    def _remove_obvious_candidates_for_value(self, value):
        for r in range(9): self._remove_candidates_from_house(value, House.ROW, r)
        for c in range(9): self._remove_candidates_from_house(value, House.COLUMN, c)
        for b in range(9): self._remove_candidates_from_house(value, House.BLOCK, b)
    def _remove_candidates_from_house(self, value, house, num):
        match house:
            case House.ROW:
                selection = self._cells_in_row(num)
            case House.COLUMN:
                selection = self._cells_in_col(num)
            case House.BLOCK:
                selection = self._cells_in_block(num)
        if np.any([x.get_value() == value for x in selection]):
            for x in selection:
                res = x.remove_candidate(value)
                #if (res):
                    #self.log.append(f"*** remove obvious candidates from {house.value} {num}, removing candidate {value} from cell [{x.row}, {x.col}]")
    def cells_in_house(self, house, num):
        match house:
            case House.ROW:
                selection = self._cells_in_row(num)
            case House.COLUMN:
                selection = self._cells_in_col(num)
            case House.BLOCK:
                selection = self._cells_in_block(num)
        return selection
    def _candidate_cells_in_house(self, values, house, num):
        return [x for x in self.cells_in_house(house, num) if x.candidates.intersection(values) == values]
    def _fill_cell(self, cell, value, house, num):
        cell.set_value(value)
        self.log.append(f"*** {house.value} {num}: filled cell [{cell.row}, {cell.col}] with value {value}")
        self._remove_obvious_candidates_for_value(value)

    def check_houses_with_single_candidate(self):
        found = False
        for v in range(1, 10):
            for r in range(9):
                candidate_cells = self._candidate_cells_in_house({v}, House.ROW, r)
                found = found or self._treat_single_candidate(candidate_cells, v, House.ROW, r)
            for c in range(9):
                candidate_cells = self._candidate_cells_in_house({v}, House.COLUMN, c)
                found = found or self._treat_single_candidate(candidate_cells, v, House.COLUMN, c)
            for b in range(9):
                candidate_cells = self._candidate_cells_in_house({v}, House.BLOCK, b)
                found = found or self._treat_single_candidate(candidate_cells, v, House.BLOCK, b)
        self.check_is_ok()
        return found
    def _treat_single_candidate(self, cells, value, house, num):
        if len(cells) != 1: return False
        self._fill_cell(cells[0], value, house, num)
        return True
        
    def check_lone_singles(self):
        found = False
        for r in range(9):
            for c in range(9):
                cell = self.cells[r,c]
                if len(cell.candidates) == 1:
                    value = next(iter(cell.candidates))
                    cell.set_value(value)
                    self.log.append(f"*** lone single: filled cell [{cell.row}, {cell.col}] with value {value}")
                    found = True
        return found

    def combinations_of_2_values(self):
        return [{v1, v2} for v1 in range(1, 10) for v2 in range(1, 10) if v2 > v1]
        return self._check_naked_combinations(combinations)
    def combinations_of_3_values(self):
        return [{v1, v2, v3} for v1 in range(1, 10) for v2 in range(1, 10) for v3 in range(1, 10) if v2 > v1 and v3 > v2]
        return self._check_naked_combinations(combinations)
    def combinations_of_4_values(self):
        return [{v1, v2, v3, v4} for v1 in range(1, 10) for v2 in range(1, 10) for v3 in range(1, 10) 
                        for v4 in range(1, 10) if v2 > v1 and v3 > v2 and v4 > v3]
    
    def check_naked_pairs(self):
        return self._check_naked_combinations(self.combinations_of_2_values())
    def check_naked_triples(self):
        combinations = [{v1, v2, v3} for v1 in range(1, 10) for v2 in range(1, 10) for v3 in range(1, 10) if v2 > v1 and v3 > v2]
        return self._check_naked_combinations(self.combinations_of_3_values())
    def check_naked_quads(self):
        return self._check_naked_combinations(self.combinations_of_4_values())
    def _check_naked_combinations(self, combinations):
        found = False
        for values in combinations:
            for r in range(9):
                candidate_cells = self._candidate_cells_in_house(values, House.ROW, r)
                found = found or self._check_naked_group(candidate_cells, values, House.ROW, r)
            for c in range(9):
                candidate_cells = self._candidate_cells_in_house(values, House.COLUMN, c)
                found = found or self._check_naked_group(candidate_cells, values, House.COLUMN, c)
            for s in range(9):
                candidate_cells = self._candidate_cells_in_house(values, House.BLOCK, s)
                found = found or self._check_naked_group(candidate_cells, values, House.BLOCK, s)
        self.check_is_ok()
        return found
    def _check_naked_group(self, selection, values, house, num):
        compatible = set([x for x in selection if x.candidates == values])
        if len(compatible) != len(values): return False
        all_cells = set(self.cells_in_house(house, num))
        to_treat = all_cells.difference(compatible)
        found = False
        for c in to_treat:
            to_maintain = c.candidates.difference(values)
            to_remove = c.candidates.difference(to_maintain)
            if to_remove:
                found = True
                self.log.append(f"*** naked group of {values} in {house.value} {num}: removed candidates {to_remove} from cell [{c.row}, {c.col}]")
                c.candidates = to_maintain.copy()
        return found

    def check_hidden_pairs(self):
        return self._check_hidden_combinations(self.combinations_of_2_values())
    def check_hidden_triples(self):
        combinations = [{v1, v2, v3} for v1 in range(1, 10) for v2 in range(1, 10) for v3 in range(1, 10) if v2 > v1 and v3 > v2]
        return self._check_hidden_combinations(self.combinations_of_3_values())
    def check_hidden_quads(self):
        return self._check_hidden_combinations(self.combinations_of_4_values())
    def _check_hidden_combinations(self, combinations):
        found = False
        for values in combinations:
            for r in range(9):
                found = found or self._check_hidden_group(self.cells_in_house(House.ROW, r), values, House.ROW, r)
            for c in range(9):
                found = found or self._check_hidden_group(self.cells_in_house(House.COLUMN, c), values, House.COLUMN, c)
            for s in range(9):
                found = found or self._check_hidden_group(self.cells_in_house(House.BLOCK, s), values, House.BLOCK, s)
        self.check_is_ok()
        return found
    def _check_hidden_group(self, selection, values, house, num):
        compatible = set([x for x in selection if x.candidates.issuperset(values)])
        if len(compatible) != len(values): return False
        others = set([x for x in selection if x.candidates.intersection(values)]) - compatible
        if others: return False
            
        found = False
        for c in compatible:
            to_remove = c.candidates.difference(values)
            if to_remove:
                found = True
                self.log.append(f"*** hidden group of {values} in {house.value} {num}: removed candidates {to_remove} from cell [{c.row}, {c.col}]")
                c.candidates = values.copy()
        return found

    def check_pointers(self):
        found = False
        for r in range(9):
            for b in range(9):
                found = found or self._check_pointers(House.ROW, r, b)
        for c in range(9):
            for b in range(9):
                found = found or self._check_pointers(House.COLUMN, c, b)
        self.check_is_ok()
        return found
    def _check_pointers(self, house, num, block):
        pointing_selection = self.cells_in_house(house, num)
        block_selection = self.cells_in_house(House.BLOCK, block)
        common_selection = set(pointing_selection) & set(block_selection)
        if not common_selection: return False
        found = False
        for v in range(1, 10):
            values = {v}
            pointing_cells = [x for x in pointing_selection if x.candidates.intersection(values) == values]
            block_cells = [x for x in block_selection if x.candidates.intersection(values) == values]
            if len(pointing_cells) == 0 or len(block_cells) == 0: continue
            pointing_cells_set = set(pointing_cells)
            block_cells_set = set(block_cells)
            common_cells_set = pointing_cells_set & block_cells_set
            if pointing_cells_set == common_cells_set and block_cells_set != common_cells_set:
                block_other_cells_set = block_cells_set - common_cells_set
                for c in block_other_cells_set:
                    self.log.append(f"*** pointing group of {v} in {house.value} {num} to block {block}: removed candidate {v} from cell [{c.row}, {c.col}]")
                    c.candidates.remove(v)
                    found = True
            if block_cells_set == common_cells_set and pointing_cells_set != common_cells_set:
                pointing_other_cells_set = pointing_cells_set - common_cells_set
                for c in pointing_other_cells_set:
                    self.log.append(f"*** pointing group of {v} in block {block} to {house.value} {num}: removed candidate {v} from cell [{c.row}, {c.col}]")
                    c.candidates.remove(v)
                    found = True
        return found

    def solve_with_brute_force(self):
        non_solved_cells = [x for x in self.cells.flatten() if x.get_value() == None]
        non_solved_cells.sort(key=lambda c: len(c.candidates))
        for cell in non_solved_cells:
            for candidate in cell.candidates:
                self.log.append(f"****** trying brute force, removing candidate {candidate} from cell [{cell.row}, {cell.col}]")
                new_board = self.clone()
                new_board.cells[cell.row, cell.col].candidates.remove(candidate)
                try:
                    self.log.append(f"****** Before brute force:\n{new_board.show(True)}")
                    solved = new_board.solve(True, False)
                except:
                  solved = False
                if (solved):
                    self.cells = new_board.cells
                    return True
        return False
            
    def not_ok_cells(self):
        return np.array([x for x in self.cells.flatten() if not x.is_ok()])
    def check_is_ok(self):
        not_ok_cells = self.not_ok_cells()
        if not_ok_cells.size == 0: return
        cells = [(x.row, x.col) for x in not_ok_cells]
        raise Exception(f"Cells in error: {cells}")        


    def is_solved(self):
        solved = np.array([x.get_value() != None for x in self.cells.flatten()])
        return solved[solved == False].size == 0
    
    def solve(self, use_brute_force = False, print_results = True):
        found = self._solve_loop()
        while found:        
            found = self._solve_loop()
        
        is_solved = self.is_solved()
        if is_solved: self.log.append(f"*** PUZZLE SOLVED WITH LOGIC")

        if not is_solved and use_brute_force:
            solved_with_brute_force = self.solve_with_brute_force()
            if solved_with_brute_force:
                self.log.append(f"*** PUZZLE SOLVED WITH BRUTE FORCE")
            else:
                self.log.append(f"*** SOLVING WITH BRUTE FORCE FAILED")
        
        is_solved = self.is_solved()
        if print_results:
            self.show_log()
            print(self.show(not is_solved))
        return is_solved
       
    def _solve_loop(self):
        found2 = False
        self.remove_obvious_candidates()
    
        found = self.check_houses_with_single_candidate()
        found2 = found2 or found
        while found:
            found = self.check_houses_with_single_candidate()
        self.remove_obvious_candidates()
        
        found = self.check_lone_singles()
        found2 = found2 or found
        while found: 
            self.remove_obvious_candidates()   
            found = self.check_lone_singles()
            
        found = self.check_naked_pairs()
        found2 = found2 or found
        while found:
            found = self.check_naked_pairs()

        found = self.check_naked_triples()
        found2 = found2 or found
        while found:
            found = self.check_naked_triples()

        found = self.check_naked_quads()
        found2 = found2 or found
        while found:
            found = self.check_naked_quads()

        found = self.check_pointers()
        found2 = found2 or found
        while found:
            found = self.check_pointers()

        found = self.check_hidden_pairs()
        found2 = found2 or found
        while found:
            found = self.check_hidden_pairs()

        found = self.check_hidden_triples()
        found2 = found2 or found
        while found:
            found = self.check_hidden_triples()

        found = self.check_hidden_quads()
        found2 = found2 or found
        while found:
            found = self.check_hidden_quads()   
            
        return found2

In [5]:
#| hide
import nbdev; nbdev.nbdev_export()